# Task 1 — Multi-Target Tracking with a Nearest-Neighbour Kalman Tracker

**UBISS 2026 · people-avoidance project · Stage 2**

This notebook **extends the course 2D Kalman filter demo** (`2d_kalman_demo.ipynb`,
[course Drive folder](https://drive.google.com/drive/folders/1zW3ADqtzvZbyUQoRqBMPrzLs0fketUs9?usp=sharing))
to: **(1.1)** multiple simulated targets and **(1.2)** a nearest-neighbour (NN)
multiple-target tracker, per `task.pdf`. It also demonstrates **Task 2.2** —
predicting future target locations by iterating the Kalman prediction step.

References: Särkkä & Svensson, *Bayesian Filtering and Smoothing*
([example notebooks](https://github.com/EEA-sensors/Bayesian-Filtering-and-Smoothing/tree/main/python/example_notebooks));
course handouts *Bayesian and Kalman filtering*, *Dynamics and measurements*,
*Multiple target tracking* (basic NN filter, pp. 31–33).

The same tracker, integrated into ROS, lives in
`src/people_avoidance/people_avoidance/tracking.py` (Task 2.1).


## Model

Wiener-velocity (constant-velocity) model, state $x = [p_x, p_y, v_x, v_y]^T$:

$$x_k = A x_{k-1} + q_{k-1}, \qquad q_{k-1} \sim N(0, Q)$$
$$y_k = H x_k + r_k, \qquad r_k \sim N(0, R)$$

$$A = \begin{pmatrix}1&0&\Delta t&0\\0&1&0&\Delta t\\0&0&1&0\\0&0&0&1\end{pmatrix}, \quad
Q = q\begin{pmatrix}\Delta t^3/3&0&\Delta t^2/2&0\\0&\Delta t^3/3&0&\Delta t^2/2\\\Delta t^2/2&0&\Delta t&0\\0&\Delta t^2/2&0&\Delta t\end{pmatrix}, \quad
H = \begin{pmatrix}1&0&0&0\\0&1&0&0\end{pmatrix}$$

Kalman filter (course form): predict $m^- = A m$, $P^- = A P A^T + Q$;
update via $v = y - Hm^-$, $S = HP^-H^T + R$, $K = P^-H^TS^{-1}$,
$m = m^- + Kv$, $P = P^- - KSK^T$.

Association uses the **squared Mahalanobis distance** (the demos own
`kalman_distance`): $d^2 = v^T S^{-1} v \sim \chi^2_2$, gated at
$\gamma = 9.21$ ($\chi^2_2$ at 99 % — with centimetre-level $R$ the 95 % gate
is only ~12 cm wide and a converging velocity estimate overshoots it).



In [ ]:
import numpy as np
import numpy.random as rd
import matplotlib.pyplot as plt
from scipy import linalg
from scipy.optimize import linear_sum_assignment

rd.seed(7)


## Helpers from the course demo (verbatim, plus multi-target simulation)


In [ ]:
# --- from 2d_kalman_demo.ipynb ---------------------------------------------
def kalman_predict(m, P, A, Q):
    m = A @ m
    P = A @ P @ A.T + Q
    return m, P

def kalman_update(m, P, H, R, y):
    S = H @ P @ H.T + R
    K = linalg.solve(S, H @ P.T, assume_a="pos").T
    m = m + K @ (y - H @ m)
    P = P - K @ S @ K.T
    return m, P

def kalman_distance(m, P, H, R, y):
    """Squared Mahalanobis distance between prediction and measurement."""
    v = y - H @ m
    S = H @ P @ H.T + R
    return np.dot(linalg.solve(S, v, assume_a="pos"), v)

def rmse(x, y):
    return np.sqrt(np.mean(np.sum(np.square(x - y), -1)))

# --- extension: simulate SEVERAL targets at once ----------------------------
def generate_targets(x0_list, A, Q, steps):
    """Simulate one CV trajectory per initial state in x0_list."""
    chol_Q = np.linalg.cholesky(Q)
    M = len(x0_list[0])
    out = []
    for x0 in x0_list:
        states = np.empty((steps, M))
        x = np.asarray(x0, float)
        for k in range(steps):
            x = A @ x + chol_Q @ rd.randn(M)
            states[k] = x
        out.append(states)
    return out                      # list of (steps, 4) arrays

def generate_measurements(true_states, H, R, p_miss=0.05, clutter_rate=1.0,
                          clutter_box=(-8, 8)):
    """Per step: noisy position of each (non-missed) target + uniform clutter.
    Returns a list over time of (n_k, 2) arrays — unlabeled, shuffled."""
    chol_R = np.linalg.cholesky(R)
    steps = true_states[0].shape[0]
    lo, hi = clutter_box
    zs = []
    for k in range(steps):
        z = [H @ s[k] + chol_R @ rd.randn(2)
             for s in true_states if rd.rand() > p_miss]
        for _ in range(rd.poisson(clutter_rate)):
            z.append(rd.uniform(lo, hi, size=2))
        rd.shuffle(z)
        zs.append(np.array(z) if z else np.empty((0, 2)))
    return zs


## Parameters (identical to the course demo)


In [ ]:
q  = 0.1      # process-noise spectral density (m^2/s^3).
              # (The course demo used q = 1.0; that diffuses a target ~18 m
              #  over 10 s — fine for one target, but unreadable for a
              #  crossing demo. q = 0.1 gives walking-like trajectories.)
dt = 0.1      # time step
s  = 0.1      # measurement noise std — LiDAR leg-centroid level.
              # (The single-target course demo used s = 0.5; with several
              #  crossing targets that noise makes association ill-posed,
              #  so we use a value representative of the real detector.)

A = np.array([[1, 0, dt, 0],
              [0, 1, 0, dt],
              [0, 0, 1, 0],
              [0, 0, 0, 1.]])
Q = q * np.array([[dt**3/3, 0, dt**2/2, 0],
                  [0, dt**3/3, 0, dt**2/2],
                  [dt**2/2, 0, dt, 0],
                  [0, dt**2/2, 0, dt]])
H = np.array([[1., 0, 0, 0],
              [0., 1, 0, 0]])
R = np.array([[s**2, 0],
              [0, s**2]])




## Task 1.1 — Multiple simulated targets

Three targets with **crossing** trajectories (the hard case for data
association), plus detection misses (5 %) and uniform clutter
(Poisson, ≈1 false measurement per scan).


In [ ]:
steps = 100
x0s = [np.array([-5., -4.,  1.2,  0.9]),     # target 0: SW -> NE
       np.array([ 5., -5., -1.1,  1.0]),     # target 1: SE -> NW
       np.array([-5.,  5.,  1.0, -1.0])]     # target 2: NW -> SE
# velocities chosen so the three paths cross PAIRWISE at different times

truth = generate_targets(x0s, A, Q, steps)
measurements = generate_measurements(truth, H, R)

fig, ax = plt.subplots(figsize=(9, 9))
for i, tr in enumerate(truth):
    ax.plot(tr[:, 0], tr[:, 1], lw=2, label=f"target {i} (truth)")
all_z = np.vstack([z for z in measurements if len(z)])
ax.scatter(all_z[:, 0], all_z[:, 1], s=8, c="red", alpha=0.3,
           label="measurements (+clutter)")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(); ax.set_title("Simulated scene")
plt.show()



## Task 1.2 — Nearest-neighbour multiple-target tracker

One Kalman filter per target (Särkkä, *Multiple target tracking* pp. 31–33):

1. **Predict** every track: $m^- = Am$, $P^- = APA^T + Q$.
2. **Cost**: $d^2_{ij} = v^T S^{-1} v$ (Mahalanobis — the demos `kalman_distance`).
3. **Gate**: forbid pairs with $d^2 > \gamma = 9.21$ ($\chi^2_2$, 99 %).
4. **Assign** jointly — `mode="hungarian"` (global NN, optimal) or
   `mode="greedy"` (classic NN: repeatedly take the smallest remaining $d^2$).
5. **Update** matched tracks; a measurement spawns a new track only if it
   fell **outside every gate** (else it is a duplicate detection); tracks
   confirm after 3 consecutive hits, die after 5 misses, and near-identical
   tracks (same position *and* velocity) are **merged**.




In [ ]:
class NNTracker:
    """NN multi-target tracker — same algorithm as the ROS tracking.py."""
    def __init__(self, A, Q, H, R, gate=9.21, max_misses=5, confirm_hits=3,
                 init_vel_std=1.5, mode="hungarian",
                 merge_dist=0.35, merge_vel=0.3):
        self.A, self.Q, self.H, self.R = A, Q, H, R
        self.gate, self.max_misses = gate, max_misses
        self.confirm_hits, self.iv = confirm_hits, init_vel_std
        self.mode = mode
        self.merge_dist, self.merge_vel = merge_dist, merge_vel
        self.tracks = []            # dicts: m P id hits misses confirmed
        self._next = 0

    def _spawn(self, z):
        self.tracks.append({
            "m": np.array([z[0], z[1], 0., 0.]),
            "P": np.diag([self.R[0,0], self.R[1,1], self.iv**2, self.iv**2]),
            "id": self._next, "hits": 0, "misses": 0, "confirmed": False})
        self._next += 1

    def step(self, zs):
        for t in self.tracks:                                   # 1. predict
            t["m"], t["P"] = kalman_predict(t["m"], t["P"], self.A, self.Q)

        nT, nZ = len(self.tracks), len(zs)
        pairs, gated_meas = [], set()
        if nT and nZ:
            BIG = 1e6                                           # 2-3. gated cost
            C = np.full((nT, nZ), BIG)
            for i, t in enumerate(self.tracks):
                for j in range(nZ):
                    d2 = kalman_distance(t["m"], t["P"], self.H, self.R, zs[j])
                    if d2 <= self.gate:
                        C[i, j] = d2
                        gated_meas.add(j)
            if self.mode == "hungarian":                        # 4. assignment
                ri, ci = linear_sum_assignment(C)
                pairs = [(i, j) for i, j in zip(ri, ci) if C[i, j] <= self.gate]
            else:                                               # greedy NN
                Cw = C.copy()
                while True:
                    i, j = np.unravel_index(np.argmin(Cw), Cw.shape)
                    if Cw[i, j] > self.gate:
                        break
                    pairs.append((int(i), int(j)))
                    Cw[i, :] = BIG + 1
                    Cw[:, j] = BIG + 1

        m_t = {i for i, _ in pairs}
        m_z = {j for _, j in pairs}
        for i, j in pairs:                                      # 5. update
            t = self.tracks[i]
            t["m"], t["P"] = kalman_update(t["m"], t["P"], self.H, self.R, zs[j])
        for i, t in enumerate(self.tracks):
            if i in m_t:
                t["misses"] = 0
                t["hits"] += 1
                t["confirmed"] |= t["hits"] >= self.confirm_hits
            else:
                t["misses"] += 1
                t["hits"] = 0
        # 6. spawn ONLY measurements outside EVERY gate (Sarkka p.33:
        #    "if no target satisfies this -> new target or outlier")
        for j in range(nZ):
            if j not in m_z and j not in gated_meas:
                self._spawn(zs[j])
        self.tracks = [t for t in self.tracks if t["misses"] <= self.max_misses]
        self._merge()                                           # 7. merge dups

    def _merge(self):
        """Same position AND same velocity = duplicate of one target.
        The velocity guard keeps crossing targets apart."""
        removed = set()
        for i in range(len(self.tracks)):
            ti = self.tracks[i]
            if ti["id"] in removed: continue
            for j in range(i + 1, len(self.tracks)):
                tj = self.tracks[j]
                if tj["id"] in removed: continue
                dp = np.hypot(ti["m"][0]-tj["m"][0], ti["m"][1]-tj["m"][1])
                dv = np.hypot(ti["m"][2]-tj["m"][2], ti["m"][3]-tj["m"][3])
                if dp >= self.merge_dist:
                    continue
                older, newer = (ti, tj) if ti["id"] < tj["id"] else (tj, ti)
                if not newer["confirmed"]:
                    removed.add(newer["id"])      # newborn duplicate: absorb
                elif dv < self.merge_vel:
                    fresher = ti if ti["misses"] <= tj["misses"] else tj
                    older["m"], older["P"] = fresher["m"].copy(), fresher["P"].copy()
                    older["misses"] = fresher["misses"]
                    older["hits"] = max(ti["hits"], tj["hits"])
                    older["confirmed"] = True
                    removed.add(newer["id"])
        if removed:
            self.tracks = [t for t in self.tracks if t["id"] not in removed]

    def confirmed(self):
        return [t for t in self.tracks if t["confirmed"]]




In [ ]:
def run_tracker(mode):
    trk = NNTracker(A, Q, H, R, mode=mode)
    history = []                       # per step: list of (id, x, y)
    for zs in measurements:
        trk.step(zs)
        history.append([(t["id"], t["m"][0], t["m"][1]) for t in trk.confirmed()])
    return trk, history

trk, history = run_tracker("hungarian")

# plot tracked trajectories per track id
by_id = {}
for k, row in enumerate(history):
    for tid, x, y in row:
        by_id.setdefault(tid, []).append((k, x, y))

fig, ax = plt.subplots(figsize=(9, 9))
for i, tr in enumerate(truth):
    ax.plot(tr[:, 0], tr[:, 1], "k--", alpha=0.5,
            label="truth" if i == 0 else None)
for tid, pts in sorted(by_id.items()):
    arr = np.array([(x, y) for _, x, y in pts])
    if len(arr) > 10:                                  # hide clutter ghosts
        ax.plot(arr[:, 0], arr[:, 1], lw=2, label=f"track #{tid}")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
ax.set_title("NN tracker (Hungarian assignment) through the 3-way crossing")
plt.show()

long_tracks = [tid for tid, pts in by_id.items() if len(pts) > steps // 2]
print(f"long-lived confirmed tracks: {len(long_tracks)} (true targets: {len(truth)})")


In [ ]:
# Tracking accuracy: RMSE of each true target vs its nearest long-lived track
for i, tr in enumerate(truth):
    best = None
    for tid in long_tracks:
        pts = {k: (x, y) for k, x, y in by_id[tid]}
        ks = [k for k in range(steps) if k in pts]
        if len(ks) < steps // 2:
            continue
        err = rmse(np.array([pts[k] for k in ks]), tr[ks, :2])
        if best is None or err < best[1]:
            best = (tid, err)
    print(f"target {i}: best track #{best[0]}  RMSE = {best[1]:.3f} m "
          f"(measurement noise s = {s} m)")


### Greedy NN vs global NN (Hungarian)

Greedy NN lets one measurement "steal" a competing tracks detection at
the crossing (the failure case from the lecture); the Hungarian algorithm
minimises the **total** cost, avoiding the swap.


In [ ]:
for mode in ("greedy", "hungarian"):
    _, hist = run_tracker(mode)
    appearances = {}
    for row in hist:
        for tid, _, _ in row:
            appearances[tid] = appearances.get(tid, 0) + 1
    long_ids = [t for t, n in appearances.items() if n > steps // 2]
    ids_mid = {tid for tid, _, _ in hist[steps // 2]}
    ids_end = {tid for tid, _, _ in hist[-1]}
    print(f"{mode:10s}: long-lived tracks = {len(long_ids)} (true: {len(truth)}), "
          f"mid-run IDs surviving to the end = {len(ids_mid & ids_end)}")



## Task 2.2 — Predicting future locations by iterating the KF prediction

Freeze the tracker at $k_0$, then iterate **only** $m \leftarrow Am$,
$P \leftarrow APA^T + Q$ for $n$ steps. The predicted mean extrapolates with
the estimated velocity, while $P$ grows — quantifying how confidence decays
with horizon. (This is exactly `KalmanTracker.predict_ahead()` in the ROS node.)


In [ ]:
k0 = 60
horizon_steps = 20                                   # 2.0 s at dt = 0.1

trk2 = NNTracker(A, Q, H, R)
for zs in measurements[:k0]:
    trk2.step(zs)

fig, ax = plt.subplots(figsize=(9, 9))
for i, tr in enumerate(truth):
    ax.plot(tr[:k0+horizon_steps, 0], tr[:k0+horizon_steps, 1], "k--", alpha=0.4)
    ax.plot(tr[k0:k0+horizon_steps, 0], tr[k0:k0+horizon_steps, 1], "g-", lw=2,
            label="true future" if i == 0 else None)

err_by_h = np.zeros(horizon_steps)
cnt_by_h = np.zeros(horizon_steps)
for t in trk2.confirmed():
    m, P = t["m"].copy(), t["P"].copy()
    pred = []
    for n in range(horizon_steps):
        m, P = kalman_predict(m, P, A, Q)
        sig = np.sqrt(np.linalg.eigvalsh(P[:2, :2])[-1])
        pred.append((m[0], m[1], sig))
    pred = np.array(pred)
    ax.plot(pred[:, 0], pred[:, 1], "r.:",
            label="KF prediction" if t is trk2.confirmed()[0] else None)
    # 1-sigma circle at the horizon end
    th = np.linspace(0, 2*np.pi, 60)
    ax.plot(pred[-1, 0] + pred[-1, 2]*np.cos(th),
            pred[-1, 1] + pred[-1, 2]*np.sin(th), "r-", alpha=0.4)
    # prediction error vs the nearest true target future
    d0 = [np.hypot(tr[k0-1, 0]-t["m"][0], tr[k0-1, 1]-t["m"][1]) for tr in truth]
    tr = truth[int(np.argmin(d0))]
    for n in range(horizon_steps):
        err_by_h[n] += np.hypot(pred[n, 0]-tr[k0+n, 0], pred[n, 1]-tr[k0+n, 1])
        cnt_by_h[n] += 1

ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
ax.set_title(f"Iterated KF prediction from k={k0} ({horizon_steps*dt:.1f} s ahead)")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, horizon_steps+1)*dt, err_by_h/np.maximum(cnt_by_h, 1), "o-")
plt.xlabel("prediction horizon (s)"); plt.ylabel("mean position error (m)")
plt.title("Prediction error grows with horizon"); plt.grid(alpha=0.3)
plt.show()


## Conclusions

- The course 2D KF demo extends naturally to N targets: **one KF per target**,
  joined by gated Mahalanobis association (the demos own `kalman_distance`).
- **Hungarian (global NN)** assignment keeps identities through the 3-way
  crossing where greedy NN can swap tracks.
- Track lifecycle (3-hit confirmation / 5-miss deletion) absorbs clutter and
  detection misses.
- Iterating the prediction step forecasts positions seconds ahead with an
  honest, growing uncertainty — used by Stage 3 to avoid people **before**
  they are in the way.

ROS integration of this exact tracker: `people_avoidance/tracking.py`
(Task 2.1); live demo with prediction ghosts: `test.py` (3D viewer).
